# 🏃 EdgeHAR — Exploratory Data Analysis

This notebook explores the UCI Human Activity Recognition dataset:
- Dataset structure and statistics
- Class distribution
- Signal visualization per activity
- Channel correlation analysis
- Statistical summary of sensor channels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully! ✅')

## 1. Load Dataset

In [ ]:
from dataset import HARDataset, CONFIG

train_dataset = HARDataset(split='train')
test_dataset = HARDataset(split='test')

class_names = CONFIG['class_names']
channel_names = [
    'Total Acc X', 'Total Acc Y', 'Total Acc Z',
    'Body Gyro X', 'Body Gyro Y', 'Body Gyro Z'
]

print(f'\nTrain samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Signal shape:  {train_dataset[0][0].shape}')
print(f'Classes:       {class_names}')

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (dataset, title) in enumerate([
    (train_dataset, 'Training Set'),
    (test_dataset, 'Test Set')
]):
    labels = dataset.labels
    unique, counts = np.unique(labels, return_counts=True)
    
    colors = sns.color_palette('husl', len(class_names))
    bars = axes[idx].bar(
        [class_names[i] for i in unique], counts,
        color=colors, edgecolor='white', linewidth=1.5
    )
    
    for bar, count in zip(bars, counts):
        axes[idx].text(
            bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(count), ha='center', fontweight='bold'
        )
    
    axes[idx].set_title(f'{title} — Class Distribution', fontweight='bold')
    axes[idx].set_ylabel('Number of Samples')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution plot saved! ✅')

## 3. Signal Visualization per Activity

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(16, 18), sharex=True)

for act_idx in range(6):
    # Find first sample of this activity
    sample_idx = np.where(train_dataset.labels == act_idx)[0][0]
    signal, _ = train_dataset[sample_idx]
    signal = signal.numpy()
    
    ax = axes[act_idx]
    for ch_idx in range(6):
        ax.plot(signal[ch_idx], label=channel_names[ch_idx], alpha=0.8, linewidth=1)
    
    ax.set_ylabel('Value')
    ax.set_title(f'{class_names[act_idx]}', fontweight='bold', loc='left')
    ax.legend(loc='upper right', fontsize=8, ncol=3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Timestep')
fig.suptitle('Raw Sensor Signals by Activity', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/signal_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Signal visualization saved! ✅')

## 4. Channel Correlation Analysis

In [ ]:
# Compute correlation between channels across all training samples
all_signals = train_dataset.signals  # (N, 6, 128)
# Flatten time dimension: (N*128, 6)
flat = all_signals.reshape(-1, 6).T  # (6, N*128)
corr_matrix = np.corrcoef(flat)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
    xticklabels=channel_names, yticklabels=channel_names,
    ax=ax, vmin=-1, vmax=1, center=0,
    linewidths=0.5, square=True
)
ax.set_title('Channel Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/channel_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlation matrix saved! ✅')

## 5. Statistical Summary

In [ ]:
all_signals = train_dataset.signals  # (N, 6, 128)

stats = []
for ch_idx, ch_name in enumerate(channel_names):
    ch_data = all_signals[:, ch_idx, :].flatten()
    stats.append({
        'Channel': ch_name,
        'Mean': f'{ch_data.mean():.4f}',
        'Std': f'{ch_data.std():.4f}',
        'Min': f'{ch_data.min():.4f}',
        'Max': f'{ch_data.max():.4f}',
        'Median': f'{np.median(ch_data):.4f}',
    })

stats_df = pd.DataFrame(stats)
print('\n📊 Channel Statistics (Training Set):')
stats_df

## 6. Summary

**Key Findings:**
- Dataset is reasonably balanced across 6 activity classes
- Dynamic activities (walking variants) show clear periodic patterns
- Static activities (sitting, standing, laying) have minimal variation
- Laying is distinguishable by gravity axis shift
- Accelerometer channels carry stronger discriminative signal than gyroscope